In [ ]:
import os
import pickle
import random
import json
import numpy as np
import pandas as pd
from PIL import Image
import ast

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from google.colab import drive

from torchinfo import summary
import dataframe_image as dfi

# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
    device = torch.device("cuda")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

# Mount Google Drive
drive.mount('/content/drive')

# Base directory for the project
BASE_DIR = '/content/drive/MyDrive/FinalGAN/'

# Dataset and embedding paths
DATA_DIR = os.path.join(BASE_DIR, 'CUB_200_2011/')
EMBEDDING_PATH_TRAIN = os.path.join(BASE_DIR, 'birds/train/')
EMBEDDING_PATH_TEST = os.path.join(BASE_DIR, 'birds/test/')
TEXT_DATA_DIR = os.path.join(BASE_DIR, 'birds/text_c10/')

# Paths for cached preprocessed dataframes ---
CACHED_DIR = os.path.join(BASE_DIR, 'cached_dataset/')
CACHED_TRAIN_DF_STAGE1_PATH = os.path.join(CACHED_DIR, 'cached_train_df_stage1.pkl')
CACHED_TEST_DF_STAGE1_PATH = os.path.join(CACHED_DIR, 'cached_test_df_stage1.pkl')
CACHED_TRAIN_DF_STAGE2_PATH = os.path.join(CACHED_DIR, 'cached_train_df_stage2.pkl')
CACHED_TEST_DF_STAGE2_PATH = os.path.join(CACHED_DIR, 'cached_test_df_stage2.pkl')

# Paths for saving generated images and model checkpoints
GENERATED_IMAGES_DIR = os.path.join(BASE_DIR, 'generated_images')
GENERATED_IMAGES_DIR_STAGE_2 = os.path.join(GENERATED_IMAGES_DIR, 'stage2')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
LOG_DIR = os.path.join(BASE_DIR, 'logs')
VISUALIZATIONS_DIR = os.path.join(BASE_DIR, 'visualizations') # New directory for visualizations

# Path for loading a specific checkpoint (set to None to find the latest)
LOAD_CHECKPOINT_PATH = None

# Image parameters
image_size_stage1 = 64
image_size_stage2 = 256
imsize = image_size_stage1

num_workers = 2
batch_size_stage1 = 64
batch_size_stage2 = 32

# Training Parameters for Stage I
num_epochs_stage1 = 600
lr_decay_step_stage1 = 50
lr_g_stage1 = 0.0002
lr_d_stage1 = 0.0002

# Training Parameters for Stage II
num_epochs_stage2 = 610
lr_decay_step_stage2 = 50
lr_g_stage2 = 0.0002
lr_d_stage2 = 0.0002

KL_WEIGHT_STAGE2 = 2.0


#Utility Functions
def load_text(filepath):
    """Loads text from a file."""
    with open(filepath, 'r') as f:
        text_data = f.read().splitlines()
    return text_data

def load_class_ids(embedding_path):
    """Loads class IDs from a pickle file."""
    with open(os.path.join(embedding_path, 'class_info.pickle'), 'rb') as f:
        class_ids = pickle.load(f, encoding='latin1')
    return class_ids

def load_embeddings(embedding_path):
    """Loads pre-computed embeddings from a pickle file."""
    with open(os.path.join(embedding_path, 'char-CNN-RNN-embeddings.pickle'), 'rb') as f:
        embeddings = pickle.load(f, encoding='latin1')
    embeddings = np.array(embeddings)
    if embeddings.ndim == 3 and embeddings.shape[1] == 10:
        original_shape = embeddings.shape
        embeddings = embeddings.reshape(-1, embeddings.shape[2])
        print(f"Flattened embeddings from {original_shape} to {embeddings.shape}")
    print(f"Loaded embeddings shape: {embeddings.shape}")
    return embeddings

def load_filenames(embedding_path):
    """Loads filenames from a pickle file and ensures they are without extensions."""
    with open(os.path.join(embedding_path, 'filenames.pickle'), 'rb') as f:
        filenames = pickle.load(f, encoding='latin1')
    processed_filenames = [os.path.splitext(fname)[0] for fname in filenames]
    return processed_filenames

def load_bounding_boxes(data_dir):
    """Loads bounding box annotations."""
    bbox_path = os.path.join(data_dir, 'bounding_boxes.txt')
    image_path = os.path.join(data_dir, 'images.txt')
    bbox_df = pd.read_csv(bbox_path, sep='\s+', header=None, names=['image_id', 'x', 'y', 'w', 'h'])
    image_df = pd.read_csv(image_path, sep='\s+', header=None, names=['image_id', 'filename'])
    merged_df = pd.merge(image_df, bbox_df, on='image_id')
    bbox_dict = {os.path.splitext(row['filename'])[0]: (row['x'], row['y'], row['w'], row['h']) for index, row in merged_df.iterrows()}
    return bbox_dict

def get_img(img_path, bbox, image_size):
    """Loads, crops, and resizes an image."""
    img = Image.open(img_path).convert('RGB')
    width, height = img.size
    r = int(bbox[0])
    c = int(bbox[1])
    re = int(bbox[2])
    ce = int(bbox[3])
    img = img.crop((r, c, r + re, c + ce))
    img = img.resize((image_size, image_size), Image.LANCZOS)
    return img

def load_dataset(data_dir, embedding_path, image_size, is_train=True, limit=None):
    """Loads the entire dataset (images, embeddings, captions), with an optional limit."""
    filenames = load_filenames(embedding_path)
    class_ids = load_class_ids(embedding_path)
    embeddings = load_embeddings(embedding_path)
    bbox_dict = load_bounding_boxes(data_dir)
    images = []
    text_comments = []
    embedding_vectors = []
    skipped_image_files = 0
    skipped_bounding_boxes = 0
    skipped_text_files = 0
    skipped_processing_errors = 0
    text_dir = TEXT_DATA_DIR
    image_extensions = ['.jpg', '.jpeg', '.png']
    if limit is not None:
        filenames = filenames[:limit]
        print(f"Limiting dataset processing to {limit} files.")
    total_files_to_process = len(filenames)
    print(f"Starting to process {total_files_to_process} files...")
    for i, filename_base in enumerate(filenames):
        if (i + 1) % 100 == 0 or (i + 1) == total_files_to_process:
            print(f"Processing image {i + 1} of {total_files_to_process}...\n")
        current_img_path = None
        img_found = False
        for ext in image_extensions:
            temp_img_path = os.path.join(data_dir, 'images', filename_base + ext)
            if os.path.exists(temp_img_path):
                current_img_path = temp_img_path
                img_found = True
                break
        if not img_found:
            skipped_image_files += 1
            continue
        bbox = bbox_dict.get(filename_base, None)
        if bbox is None:
            skipped_bounding_boxes += 1
            continue
        text_file_path = os.path.join(text_dir, filename_base + '.txt')
        current_caption = ""
        current_embedding_vector = None
        if os.path.exists(text_file_path):
            try:
                captions = load_text(text_file_path)
                current_caption = random.choice(captions)
                num_captions_per_image = 10
                start_idx_for_embeddings = i * num_captions_per_image
                end_idx_for_embeddings = start_idx_for_embeddings + num_captions_per_image
                if len(embeddings) >= end_idx_for_embeddings:
                    rand_embed_idx = random.randint(start_idx_for_embeddings, end_idx_for_embeddings - 1)
                    current_embedding_vector = embeddings[rand_embed_idx]
                elif i < len(embeddings):
                    current_embedding_vector = embeddings[i]
                else:
                    print(f"Warning: No valid embedding found for {filename_base} at index {i}. Skipping sample.")
                    skipped_processing_errors += 1
                    continue
            except Exception as e:
                print(f"Error loading text or selecting embedding for {filename_base}: {e}. Skipping sample.")
                skipped_processing_errors += 1
                continue
        else:
            skipped_text_files += 1
            current_embedding_vector = np.zeros(embeddings.shape[1])
        try:
            img = get_img(current_img_path, bbox, image_size)
            images.append(np.array(img))
            embedding_vectors.append(current_embedding_vector)
            text_comments.append(current_caption)
        except Exception as e:
            print(f"Error processing image {current_img_path}: {e}. Skipping sample.")
            skipped_processing_errors += 1
            continue
    print(f"\n--- Data Loading Summary ({'Train' if is_train else 'Test'}) ---")
    print(f"Total filenames to process: {total_files_to_process}")
    print(f"Samples loaded successfully: {len(images)}")
    print(f"Skipped due to missing image files: {skipped_image_files}")
    print(f"Skipped due to missing bounding boxes: {skipped_bounding_boxes}")
    print(f"Samples with missing text files (empty caption/zero embedding used): {skipped_text_files}")
    print(f"Skipped due to other processing errors: {skipped_processing_errors}")
    print("--------------------------------------")
    if not (len(images) == len(embedding_vectors) == len(text_comments)):
        print(f"CRITICAL ERROR: Mismatched lengths before DataFrame creation!")
        print(f"Images: {len(images)}, Embeddings: {len(embedding_vectors)}, Comments: {len(text_comments)}")
        raise ValueError("Data lists have inconsistent lengths. Cannot create DataFrame.")
    data_df = pd.DataFrame({
        'image': images,
        'embedding': embedding_vectors,
        'comment': text_comments
    })
    return data_df


# Custom PyTorch Dataset
class MyDataset(Dataset):
    def __init__(self, df, image_size, transform=None):
        self.df = df
        self.image_size = image_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        item = self.df.iloc[idx]
        img_np = item['image']
        embedding = item['embedding']
        comment = item['comment']
        img = Image.fromarray(np.uint8(img_np))
        if self.transform:
            img = self.transform(img)
        embedding = torch.from_numpy(embedding).float()
        return img, embedding, comment

# Image Transformations
transform_stage1 = transforms.Compose([
    transforms.RandomCrop(image_size_stage1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_stage2 = transforms.Compose([
    transforms.RandomCrop(image_size_stage2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


# Network Architectures
def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)

class upBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(upBlock, self).__init__()
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.conv = conv3x3(in_channels, out_channels)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(True)
    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class CA_NET(nn.Module):
    def __init__(self, t_dim=1024, c_dim=256):
        super(CA_NET, self).__init__()
        self.t_dim = t_dim
        self.c_dim = c_dim
        self.fc = nn.Linear(self.t_dim, self.c_dim * 2)
        self.relu = nn.ReLU(True)
    def encode(self, text_embedding):
        x = self.relu(self.fc(text_embedding))
        mu = x[:, :self.c_dim]
        logvar = x[:, self.c_dim:]
        return mu, logvar
    def reparametrize(self, mu, logvar):
        std = logvar.mul(0.5).exp_()
        eps = torch.randn_like(std)
        return eps.mul(std).add_(mu)
    def forward(self, text_embedding):
        mu, logvar = self.encode(text_embedding)
        c_code = self.reparametrize(mu, logvar)
        return c_code, mu, logvar

class STAGE1_G(nn.Module):
    def __init__(self):
        super(STAGE1_G, self).__init__()
        self.gf_dim = 128
        self.ef_dim = 1024
        self.z_dim = 100
        self.c_dim = 256
        self.ca_net = CA_NET(self.ef_dim, self.c_dim)
        self.fc = nn.Sequential(
            nn.Linear(self.z_dim + self.c_dim, self.gf_dim * 8 * 4 * 4, bias=False),
            nn.BatchNorm1d(self.gf_dim * 8 * 4 * 4),
            nn.ReLU(True)
        )
        self.upsample1 = upBlock(self.gf_dim * 8, self.gf_dim * 4)
        self.upsample2 = upBlock(self.gf_dim * 4, self.gf_dim * 2)
        self.upsample3 = upBlock(self.gf_dim * 2, self.gf_dim)
        self.upsample4 = upBlock(self.gf_dim, self.gf_dim // 2)
        self.img_conv = nn.Sequential(
            conv3x3(self.gf_dim // 2, 3),
            nn.Tanh()
        )
    def forward(self, text_embedding, noise):
        c_code, mu, logvar = self.ca_net(text_embedding)
        z_c_code = torch.cat((noise, c_code), 1)
        h_code = self.fc(z_c_code)
        h_code = h_code.view(-1, self.gf_dim * 8, 4, 4)
        h_code = self.upsample1(h_code)
        h_code = self.upsample2(h_code)
        h_code = self.upsample3(h_code)
        h_code = self.upsample4(h_code)
        fake_img = self.img_conv(h_code)
        return fake_img, mu, logvar

class STAGE1_D(nn.Module):
    def __init__(self):
        super(STAGE1_D, self).__init__()
        self.df_dim = 64
        self.ef_dim = 1024
        self.c_dim = 256
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, self.df_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.df_dim, self.df_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 2),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(self.df_dim * 2, self.df_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(self.df_dim * 4, self.df_dim * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.ca_net_d = CA_NET(self.ef_dim, self.c_dim)
        self.get_logits = D_GET_LOGITS(self.df_dim * 8, self.c_dim)
    def forward(self, image, text_embedding):
        h_code = self.conv1(image)
        h_code = self.conv2(h_code)
        h_code = self.conv3(h_code)
        h_code = self.conv4(h_code)
        c_code_d, _, _ = self.ca_net_d(text_embedding)
        output = self.get_logits(h_code, c_code_d)
        return output

class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv1 = conv3x3(channels, channels)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(True)
        self.conv2 = conv3x3(channels, channels)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class STAGE2_G(nn.Module):
    def __init__(self, STAGE1_G_model):
        super(STAGE2_G, self).__init__()
        self.gf_dim = 128 * 2
        self.ef_dim = 1024
        self.c_dim = 256
        self.STAGE1_G = STAGE1_G_model
        for param in self.STAGE1_G.parameters():
            param.requires_grad = False
        self.ca_net = CA_NET(self.ef_dim, self.c_dim)
        self.conv_init = nn.Sequential(
            conv3x3(3, self.gf_dim),
            nn.BatchNorm2d(self.gf_dim),
            nn.ReLU(True)
        )
        self.text_img_concat = nn.Sequential(
            conv3x3(self.gf_dim + self.c_dim, self.gf_dim),
            nn.BatchNorm2d(self.gf_dim),
            nn.ReLU(True)
        )
        self.res_block1 = ResBlock(self.gf_dim)
        self.res_block2 = ResBlock(self.gf_dim)
        self.upsample1 = upBlock(self.gf_dim, self.gf_dim // 2)
        self.upsample2 = upBlock(self.gf_dim // 2, self.gf_dim // 4)
        self.img_conv = nn.Sequential(
            conv3x3(self.gf_dim // 4, 3),
            nn.Tanh()
        )
    def forward(self, text_embedding, noise):
        self.STAGE1_G.eval()
        with torch.no_grad():
            fake_img_stage1, _, _ = self.STAGE1_G(text_embedding, noise)
        self.STAGE1_G.train()
        c_code, mu, logvar = self.ca_net(text_embedding)
        h_code = self.conv_init(fake_img_stage1)
        c_code_expanded = c_code.view(-1, c_code.size(1), 1, 1).repeat(1, 1, h_code.size(2), h_code.size(3))
        h_c_code = torch.cat((h_code, c_code_expanded), 1)
        h_c_code = self.text_img_concat(h_c_code)
        h_c_code = self.res_block1(h_c_code)
        h_c_code = self.res_block2(h_c_code)
        h_c_code = self.upsample1(h_c_code)
        h_c_code = self.upsample2(h_c_code)
        fake_img_stage2 = self.img_conv(h_c_code)
        return fake_img_stage2, mu, logvar

class STAGE2_D(nn.Module):
    def __init__(self):
        super(STAGE2_D, self).__init__()
        self.df_dim = 64
        self.ef_dim = 1024
        self.c_dim = 256
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, self.df_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.df_dim, self.df_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 2),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(self.df_dim * 2, self.df_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(self.df_dim * 4, self.df_dim * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(self.df_dim * 8, self.df_dim * 16, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 16),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv6 = nn.Sequential(
            nn.Conv2d(self.df_dim * 16, self.df_dim * 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.df_dim * 32),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.ca_net_d = CA_NET(self.ef_dim, self.c_dim)
        self.get_logits = D_GET_LOGITS(self.df_dim * 32, self.c_dim)
    def forward(self, image, text_embedding):
        h_code = self.conv1(image)
        h_code = self.conv2(h_code)
        h_code = self.conv3(h_code)
        h_code = self.conv4(h_code)
        h_code = self.conv5(h_code)
        h_code = self.conv6(h_code)
        c_code_d, _, _ = self.ca_net_d(text_embedding)
        output = self.get_logits(h_code, c_code_d)
        return output

# KL Divergence Loss
def KL_loss(mu, logvar):
    KLD_element = mu.pow(2).add_(logvar.exp()).mul_(-1).add_(1).add_(logvar)
    KLD = torch.mean(KLD_element).mul_(-0.5)
    return KLD

# Weights Initialization
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Visualization Functions
def save_model_summary_image(model, input_size, save_path, stage_num, model_type):
    """Generates a model summary and saves it as an image."""
    try:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        with open(f"{save_path[:-4]}.txt", 'w') as f:
            if model_type == 'G':
                summary_output = summary(model, input_size=[(1, 1024), (1, 100)], depth=10)
            else:
                summary_output = summary(model, input_size=[(1, 3, input_size, input_size), (1, 1024)], depth=10)
            f.write(str(summary_output))

        # Read the text file and convert to an image
        fig = plt.figure(figsize=(12, 12))
        ax = fig.add_subplot(111)
        ax.axis('off')

        with open(f"{save_path[:-4]}.txt", 'r') as f:
            text = f.read()
            ax.text(0.01, 0.99, text, transform=ax.transAxes, fontsize=8, family='monospace',
                    verticalalignment='top', horizontalalignment='left')

        plt.title(f"Stage {stage_num} {model_type} Network Architecture Summary", fontsize=16)
        plt.savefig(save_path)
        plt.close()

        print(f"Saved Stage {stage_num} {model_type} network architecture to {save_path}")
    except Exception as e:
        print(f"Error generating model architecture image for Stage {stage_num} {model_type}: {e}")
        print("Falling back to saving as text file.")

def save_trainable_parameters_table(models, stage_num, save_dir):
    """
    Creates and saves a table of trainable parameters for each model.
    """
    os.makedirs(save_dir, exist_ok=True)
    df_list = []
    for name, model in models.items():
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        data = {'Layer Name': [], 'Parameter Count': [], 'Trainable': []}
        for n, p in model.named_parameters():
            data['Layer Name'].append(n)
            data['Parameter Count'].append(p.numel())
            data['Trainable'].append(p.requires_grad)

        df = pd.DataFrame(data)
        df_list.append(df)

    final_df = pd.concat(df_list, keys=[f"Stage {stage_num} {name}" for name in models.keys()])
    final_df.index.names = ['Model', '']

    output_path = os.path.join(save_dir, f'stage{stage_num}_trainable_parameters_table.png')
    dfi.export(final_df, output_path, table_conversion='matplotlib', max_rows=-1, max_cols=-1)
    print(f"Saved Stage {stage_num} trainable parameters table to {output_path}")

def save_preprocessed_images(dataloader, save_dir, stage_num):
    """Saves a batch of preprocessed images from the dataloader."""
    os.makedirs(save_dir, exist_ok=True)
    if not dataloader:
        print(f"Skipping saving preprocessed images for Stage {stage_num} as dataloader is empty.")
        return
    try:
        images, _, _ = next(iter(dataloader))
        images = images[:16]
        grid = make_grid(images, nrow=4, padding=2, normalize=True, scale_each=True)
        save_path = os.path.join(save_dir, f'stage{stage_num}_preprocessed_images.png')
        save_image(grid, save_path)
        print(f"Saved Stage {stage_num} preprocessed images to {save_path}")
    except Exception as e:
        print(f"Error saving preprocessed images for Stage {stage_num}: {e}")

# Main Execution Block
print("Starting script execution...")
os.makedirs(VISUALIZATIONS_DIR, exist_ok=True)

# Stage I Network and Data Setup
print("\n--- Setting up Stage I Networks and Data ---")
netG_stage1 = STAGE1_G().to(device)
netD_stage1 = STAGE1_D().to(device)
print("Checking for cached Stage I training data...")
if os.path.exists(CACHED_TRAIN_DF_STAGE1_PATH):
    print(f"Loading cached training data from {CACHED_TRAIN_DF_STAGE1_PATH}...")
    train_df_stage1 = pd.read_pickle(CACHED_TRAIN_DF_STAGE1_PATH)
    print(f"Cached training data loaded: {len(train_df_stage1)} samples.")
else:
    print("Cached training data not found. Re-creating and caching...")
    train_df_stage1 = load_dataset(DATA_DIR, EMBEDDING_PATH_TRAIN, image_size_stage1, is_train=True, limit=batch_size_stage1 * 2)
    os.makedirs(os.path.dirname(CACHED_TRAIN_DF_STAGE1_PATH), exist_ok=True)
    train_df_stage1.to_pickle(CACHED_TRAIN_DF_STAGE1_PATH)
    print("Training data cached.")
if len(train_df_stage1) > 0:
    train_dataset_stage1 = MyDataset(train_df_stage1, image_size_stage1, transform_stage1)
    train_dataloader_stage1 = DataLoader(train_dataset_stage1, batch_size=batch_size_stage1, shuffle=True, num_workers=num_workers)
    print("Stage I Train DataLoader created.")
else:
    train_dataloader_stage1 = None
    print("Warning: Stage I Training dataset is empty.")

# Save Stage I Visualizations
print("\nSaving Stage I network visualizations and preprocessed images...")
save_model_summary_image(netG_stage1, input_size=image_size_stage1, save_path=os.path.join(VISUALIZATIONS_DIR, 'stage1_G_arch.png'), stage_num=1, model_type='G')
save_model_summary_image(netD_stage1, input_size=image_size_stage1, save_path=os.path.join(VISUALIZATIONS_DIR, 'stage1_D_arch.png'), stage_num=1, model_type='D')
save_preprocessed_images(train_dataloader_stage1, VISUALIZATIONS_DIR, 1)
save_trainable_parameters_table({'G': netG_stage1, 'D': netD_stage1}, 1, VISUALIZATIONS_DIR)
print("Stage I visualizations saved.")

# Stage II Network and Data Setup
print("\n--- Setting up Stage II Networks and Data ---")
netG_stage2 = STAGE2_G(netG_stage1).to(device)
netD_stage2 = STAGE2_D().to(device)
print("Checking for cached Stage II training data...")
if os.path.exists(CACHED_TRAIN_DF_STAGE2_PATH):
    print(f"Loading cached training data for Stage II from {CACHED_TRAIN_DF_STAGE2_PATH}...")
    train_df_stage2 = pd.read_pickle(CACHED_TRAIN_DF_STAGE2_PATH)
    print(f"Cached Stage II training data loaded: {len(train_df_stage2)} samples.")
else:
    print("Cached Stage II training data not found. Re-creating and caching...")
    train_df_stage2 = load_dataset(DATA_DIR, EMBEDDING_PATH_TRAIN, image_size_stage2, is_train=True, limit=batch_size_stage2 * 2)
    os.makedirs(os.path.dirname(CACHED_TRAIN_DF_STAGE2_PATH), exist_ok=True)
    train_df_stage2.to_pickle(CACHED_TRAIN_DF_STAGE2_PATH)
    print("Stage II training data cached.")
if len(train_df_stage2) > 0:
    train_dataset_stage2 = MyDataset(train_df_stage2, image_size_stage2, transform_stage2)
    train_dataloader_stage2 = DataLoader(train_dataset_stage2, batch_size=batch_size_stage2, shuffle=True, num_workers=num_workers)
    print("Stage II Train DataLoader created.")
else:
    train_dataloader_stage2 = None
    print("Warning: Stage II Training dataset is empty.")

# Save Stage II Visualizations
print("\nSaving Stage II network visualizations and preprocessed images...")
save_model_summary_image(netG_stage2, input_size=image_size_stage2, save_path=os.path.join(VISUALIZATIONS_DIR, 'stage2_G_arch.png'), stage_num=2, model_type='G')
save_model_summary_image(netD_stage2, input_size=image_size_stage2, save_path=os.path.join(VISUALIZATIONS_DIR, 'stage2_D_arch.png'), stage_num=2, model_type='D')
save_preprocessed_images(train_dataloader_stage2, VISUALIZATIONS_DIR, 2)
save_trainable_parameters_table({'G': netG_stage2, 'D': netD_stage2}, 2, VISUALIZATIONS_DIR)
print("Stage II visualizations saved.")

print("\n--- All requested visualizations have been saved. No training was performed. ---")

<>:122: SyntaxWarning: invalid escape sequence '\s'
<>:123: SyntaxWarning: invalid escape sequence '\s'
<>:122: SyntaxWarning: invalid escape sequence '\s'
<>:123: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-639794574.py:122: SyntaxWarning: invalid escape sequence '\s'
  bbox_df = pd.read_csv(bbox_path, sep='\s+', header=None, names=['image_id', 'x', 'y', 'w', 'h'])
/tmp/ipython-input-639794574.py:123: SyntaxWarning: invalid escape sequence '\s'
  image_df = pd.read_csv(image_path, sep='\s+', header=None, names=['image_id', 'filename'])


CUDA is available. Using GPU.
Mounted at /content/drive
Starting script execution...

--- Setting up Stage I Networks and Data ---
Checking for cached Stage I training data...
Loading cached training data from /content/drive/MyDrive/FinalGAN/cached_dataset/cached_train_df_stage1.pkl...
Cached training data loaded: 8855 samples.
Stage I Train DataLoader created.

Saving Stage I network visualizations and preprocessed images...
Saved Stage 1 G network architecture to /content/drive/MyDrive/FinalGAN/visualizations/stage1_G_arch.png
Saved Stage 1 D network architecture to /content/drive/MyDrive/FinalGAN/visualizations/stage1_D_arch.png
Saved Stage 1 preprocessed images to /content/drive/MyDrive/FinalGAN/visualizations/stage1_preprocessed_images.png
Saved Stage 1 trainable parameters table to /content/drive/MyDrive/FinalGAN/visualizations/stage1_trainable_parameters_table.png
Stage I visualizations saved.

--- Setting up Stage II Networks and Data ---
Checking for cached Stage II training d